# 02. 모델(LLM) — LangChain V1 모델 완전 가이드

> LangChain V1의 모델 계층은 `init_chat_model("provider:model")` 한 줄로 통일됐어요. invoke/stream/batch와 Content Blocks까지 모델을 다루는 패턴을 정리해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `init_chat_model`로 20가지 이상의 제공자를 동일한 방식으로 초기화할 수 있어요
2. `invoke`, `stream`, `ainvoke`, `astream`, `batch` 5가지 호출 방식을 상황에 맞게 선택할 수 있어요
3. `bind_tools`와 `with_structured_output`으로 Tool Calling과 구조화 출력을 구현할 수 있어요
4. `UsageMetadataCallbackHandler`로 토큰 사용량을 추적하고 비용을 관리할 수 있어요
5. Logprobs와 멀티모달 입력을 활용할 수 있어요
6. OpenAI 호환 API로 커스텀 엔드포인트와 사내 모델 서버를 연결할 수 있어요

## 사전 지식

- LangChain V0 기본 사용 경험
- 이전 노트북: `01-LangGraph-Python-Basics.ipynb` — TypedDict, Annotated, add_messages

## LLM과 LangChain V1 모델 아키텍처

LLM(Large Language Model)은 에이전트의 추론 엔진이에요. LangChain V1에서는 모든 제공자를 `init_chat_model` 하나로 통합했어요.

```mermaid
flowchart LR
    A["사용자 입력<br>User Input"] --> B["init_chat_model<br>'provider:model'"]:::process
    B --> C{"호출 방식<br>Call Mode"}:::process
    C --> D["invoke<br>동기 전체 응답"]:::output
    C --> E["stream<br>실시간 스트리밍"]:::output
    C --> F["ainvoke / astream<br>비동기 처리"]:::output
    C --> G["batch<br>병렬 일괄 처리"]:::output
    D --> H["AIMessage<br>응답 객체"]:::storage
    E --> H
    F --> H
    G --> H
    H --> I["Tool Calling<br>도구 호출"]:::process
    H --> J["Structured Output<br>구조화 출력"]:::process

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### 핵심 구성 요소

| 구성 요소 | 설명 | V1 변경 사항 |
|-----------|------|--------------|
| `init_chat_model` | 통합 모델 초기화 함수 | `ChatOpenAI`, `ChatAnthropic` 직접 임포트 불필요 |
| `provider:model` 형식 | `"openai:gpt-4o-mini"` 처럼 간결하게 지정 | 제공자 자동 감지 |
| `AIMessage` | 모든 응답의 표준 객체 | `content_blocks` 속성 추가 |
| `UsageMetadataCallbackHandler` | 토큰 사용량 누적 추적 | 모델별 분리 집계 지원 |

> 🔑 **핵심 개념**: LangChain V1에서는 `from langchain.chat_models import init_chat_model`로 모든 제공자를 통일된 방식으로 사용해요. 제공자별 패키지를 직접 import할 필요가 없어서 코드가 훨씬 간결해졌어요.

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 설정
# ---------------------------------------------------
# dotenv: .env 파일에서 API 키를 로드해요
from dotenv import load_dotenv
import os

# .env 파일의 환경 변수를 로드해요
load_dotenv(override=True)

# LangSmith 추적 설정 (선택 사항)
# LangSmith에서 모델 호출 과정을 시각적으로 디버깅할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Part02-Models"

True

## 1. 모델 초기화: init_chat_model

LangChain V1의 핵심 변화 중 하나는 `init_chat_model` 함수예요. `provider:model` 형식의 문자열 하나로 20가지 이상의 제공자를 동일하게 초기화할 수 있어요.

> 🎯 **강의 포인트**: V0에서는 `ChatOpenAI`, `ChatAnthropic` 등을 각각 import했어요. V1에서는 `init_chat_model` 하나로 통합되었어요. 제공자를 바꿀 때 import 문을 수정할 필요가 없어서 유지보수가 훨씬 편해졌어요.

### 지원 제공자 목록

| 제공자 | 형식 | 패키지 |
|--------|------|---------|
| OpenAI | `openai:gpt-4o-mini` | `langchain-openai` |
| Anthropic | `anthropic:claude-sonnet-4-5` | `langchain-anthropic` |
| Google Gemini | `google_genai:gemini-2.0-flash` | `langchain-google-genai` |
| Groq | `groq:llama-3.3-70b-versatile` | `langchain-groq` |
| AWS Bedrock | `bedrock:us.amazon.nova-lite-v1:0` | `langchain-aws` |
| Mistral | `mistralai:mistral-large-latest` | `langchain-mistralai` |
| DeepSeek | `deepseek:deepseek-chat` | `langchain-deepseek` |

In [2]:
# ---------------------------------------------------
# 모델 초기화: init_chat_model
# ---------------------------------------------------
# V1 통합 임포트 경로 사용 (langchain_core가 아닌 langchain)
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델로 교체할 때는 문자열만 바꾸면 돼요!
# 예: "anthropic:claude-sonnet-4-5"
# 예: "google_genai:gemini-2.0-flash"
# 예: "groq:llama-3.3-70b-versatile"
model = init_chat_model("openai:gpt-4o-mini")

print(f"모델 타입: {type(model).__name__}")
print(f"모델 이름: {model.model_name}")

모델 타입: ChatOpenAI
모델 이름: gpt-4o-mini


In [3]:
# ---------------------------------------------------
# 모델 세부 설정
# ---------------------------------------------------
# 더 세밀한 제어가 필요할 때는 키워드 인자를 추가해요
from langchain.chat_models import init_chat_model

model_detailed = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0.1,   # 응답의 무작위성: 0(결정적) ~ 2(창의적)
    max_tokens=500,    # 최대 생성 토큰 수
    timeout=30,        # 요청 타임아웃(초)
    max_retries=2,     # 실패 시 최대 재시도 횟수
)

# 세부 설정 모델 생성 완료
print(f"temperature: {model_detailed.temperature}")

temperature: 0.1


## 2. 기본 호출: invoke()

`invoke()`는 동기(synchronous) 방식으로 전체 응답이 생성될 때까지 기다렸다가 한 번에 반환해요.

### 왜 invoke()부터 배우나요?

`invoke()`는 가장 단순하고 직관적인 호출 방식이에요. 전화를 걸어서 상대방이 말을 다 끝낼 때까지 기다렸다가 한 번에 듣는 것과 같아요. 반면 `stream()`은 실시간 통역처럼 한 마디씩 즉시 전달받는 방식이에요. 먼저 단순한 `invoke()`를 이해하고 나면 다른 방식들이 자연스럽게 이해돼요.

메시지는 두 가지 방식으로 전달할 수 있어요:
- **튜플 형식**: `("role", "content")` — 간결하고 직관적이에요
- **메시지 객체**: `HumanMessage("...")` — 명시적이고 타입 안전해요

> 🔑 **핵심 개념**: `invoke()`의 응답은 항상 `AIMessage` 객체예요. `response.content`로 텍스트를, `response.usage_metadata`로 토큰 사용량을 확인할 수 있어요.

In [4]:
# ---------------------------------------------------
# invoke(): 동기 방식 전체 응답 받기
# ---------------------------------------------------
# 튜플 형식으로 메시지 전달
# (role, content) 형식: role은 "system", "human", "ai" 중 하나예요
messages = [
    ("system", "You are a helpful assistant. Answer in Korean."),
    ("human", "대한민국의 수도는 어디야?"),
]

# invoke 호출: 전체 응답이 생성될 때까지 기다려요
response = model.invoke(messages)
print(response)

content='대한민국의 수도는 서울입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 29, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ed279b101f', 'id': 'chatcmpl-DZSPZ9U7JeLcUlE4mXzYhPQgsVOI4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019dd1e6-ebe2-7bc3-81f6-d2312d86447f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 29, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [5]:
# ---------------------------------------------------
# AIMessage 응답 구조 살펴보기
# ---------------------------------------------------
# 텍스트 답변만 추출
# === 텍스트 답변 ===
print(response.content)
print()

# 토큰 사용량 정보 (비용 계산에 필요해요)
# === 토큰 사용량 ===
print(response.usage_metadata)
print()

# 응답 메타데이터 (모델명, 종료 사유 등)
# === 응답 메타데이터 ===
print(response.response_metadata)

대한민국의 수도는 서울입니다.

{'input_tokens': 29, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

{'token_usage': {'completion_tokens': 8, 'prompt_tokens': 29, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ed279b101f', 'id': 'chatcmpl-DZSPZ9U7JeLcUlE4mXzYhPQgsVOI4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}


## 3. 스트리밍: stream()

`stream()`은 토큰이 생성되는 즉시 출력해요. 긴 응답을 기다리지 않아도 되므로 사용자 경험이 크게 향상돼요.

> 💡 **실무 팁**: 챗봇이나 사용자 인터페이스에서는 항상 `stream()`을 사용하세요. 첫 번째 토큰이 나타나는 시간(TTFT: Time To First Token)이 짧아져 사용자가 응답이 진행 중임을 바로 알 수 있어요.

> ⚠️ **자주 하는 실수**: `stream()`이 반환하는 것은 generator예요. `for` 루프 없이 바로 출력하면 generator 객체가 출력돼요. 반드시 루프로 순회해야 해요.

In [6]:
# ---------------------------------------------------
# stream(): 실시간 스트리밍 출력
# ---------------------------------------------------
# stream()은 generator를 반환해요
# for 루프로 각 청크(chunk)를 순회하면서 즉시 출력해요
# 스트리밍 출력:
for chunk in model.stream(messages):
    # end=""로 줄바꿈 없이, flush=True로 버퍼를 즉시 비워요
    print(chunk.content, end="", flush=True)
print()  # 마지막 줄바꿈

대한민국의 수도는 서울입니다.


In [7]:
# ---------------------------------------------------
# 스트리밍 헬퍼 함수 직접 구현
# ---------------------------------------------------
# 스트리밍 응답을 직접 구현해요
# 내부 동작을 이해하는 것이 중요해요!

def stream_model_response(stream_generator) -> str:
    """모델 스트림을 출력하고 전체 응답을 반환하는 헬퍼 함수예요.
    
    Args:
        stream_generator: model.stream()이 반환하는 generator
    
    Returns:
        전체 응답 텍스트
    """
    full_response = ""
    for chunk in stream_generator:
        content = chunk.content
        print(content, end="", flush=True)
        full_response += content
    print()  # 마지막 줄바꿈
    return full_response


# 헬퍼 함수 사용
question = [("human", "파이썬의 장점을 3가지만 간단히 말해줘.")]
full_text = stream_model_response(model.stream(question))
print(f"\n총 응답 길이: {len(full_text)}자")

파이썬의 장점 3가지는 다음과 같습니다:

1. **쉬운 문법**: 파이썬은 읽기 쉽고 배우기 쉬운 문법을 가지고 있어, 초보자부터 전문가까지 모두가 쉽게 접근할 수 있습니다.

2. **풍부한 라이브러리**: 다양한 분야에 걸쳐 많은 라이브러리와 프레임워크가 제공되어, 데이터 분석, 웹 개발, 머신러닝 등 여러 작업을 효율적으로 수행할 수 있습니다.

3. **커뮤니티와 지원**: 파이썬은 큰 커뮤니티가 있어서, 대부분의 문제에 대한 해결책이나 지원을 쉽게 찾을 수 있으며, 다양한 자료와 튜토리얼이 있습니다.

총 응답 길이: 288자


## 4. 비동기 처리: ainvoke()와 astream()

Python의 `async/await` 구문으로 비동기 처리를 해요. 여러 모델 호출을 동시에 처리하거나 I/O 바운드 작업과 병렬로 실행할 때 유용해요.

> 🔑 **핵심 개념**: Jupyter 노트북은 기본적으로 이벤트 루프가 실행 중이에요. 그래서 `await`를 최상위 수준에서 바로 사용할 수 있어요. Python 스크립트에서는 `asyncio.run()`으로 감싸야 해요.

In [8]:
# ---------------------------------------------------
# ainvoke(): 비동기 동기 호출
# ---------------------------------------------------
# await 키워드로 비동기 응답을 기다려요
# Jupyter에서는 최상위 수준에서 await 사용 가능해요
response_async = await model.ainvoke(messages)
# 비동기 응답:
print(response_async.content)

대한민국의 수도는 서울입니다.


In [9]:
# ---------------------------------------------------
# astream(): 비동기 스트리밍
# ---------------------------------------------------
# async for 구문으로 비동기 스트림을 순회해요
# 비동기 컨텍스트에서 스트리밍할 때 사용해요
# 비동기 스트리밍 출력:
async for chunk in model.astream(messages):
    print(chunk.content, end="", flush=True)
print()

대한민국의 수도는 서울입니다.


## 5. 배치 처리: batch()

`batch()`는 여러 요청을 한 번에 병렬로 처리해요. 다량의 데이터를 처리할 때 순차 호출보다 훨씬 효율적이에요.

> 💡 **실무 팁**: 대량 텍스트 분류, 번역, 요약 등에서 `batch()`를 사용하면 처리 시간을 크게 단축할 수 있어요. 내부적으로 스레드 풀을 사용해서 병렬 처리해요.

### batch() vs batch_as_completed() 비교

| 구분 | `batch()` | `batch_as_completed()` |
|------|-----------|----------------------|
| 반환 순서 | 입력 순서와 동일 | 완료된 순서대로 |
| 반환 형태 | `list[AIMessage]` | `Iterator[(index, AIMessage)]` |
| 사용 시점 | 순서가 중요할 때 | 빠른 응답부터 처리할 때 |
| 속도 | 가장 느린 요청 기준 | 완료 즉시 반환 |

In [10]:
# ---------------------------------------------------
# batch(): 여러 요청을 병렬로 처리
# ---------------------------------------------------
# 리스트로 여러 입력을 전달해요
# 입력 순서와 동일한 순서로 응답을 반환해요
questions = [
    "대한민국의 수도는?",
    "캐나다의 수도는?",
    "브라질의 수도는?",
    "호주의 수도는?",
]

# 병렬로 4개 요청을 처리해요
responses = model.batch(questions)

# 배치 처리 결과:
for i, (q, r) in enumerate(zip(questions, responses), 1):
    print(f"{i}. 질문: {q}")
    print(f"   답변: {r.content}")

1. 질문: 대한민국의 수도는?
   답변: 대한민국의 수도는 서울입니다.
2. 질문: 캐나다의 수도는?
   답변: 캐나다의 수도는 오타와(Ottawa)입니다.
3. 질문: 브라질의 수도는?
   답변: 브라질의 수도는 브라질리아입니다. 1960년에 수도로 지정되었으며, 현대적인 계획 도시로 알려져 있습니다.
4. 질문: 호주의 수도는?
   답변: 호주의 수도는 캔버라(Canberra)입니다. 캔버라는 호주 내륙에 위치하며, 국가의 정치 중심지로서 여러 정부 기관과 외교 공관이 있습니다.


In [11]:
# ---------------------------------------------------
# batch_as_completed(): 완료 순서대로 결과 반환
# ---------------------------------------------------
# 응답이 완료되는 순서대로 (인덱스, 결과) 쌍을 반환해요
# 순서가 중요하지 않고 빠른 응답을 먼저 처리하고 싶을 때 유용해요
# batch_as_completed() 결과 (완료 순서):
for idx, result in model.batch_as_completed(questions):
    print(f"  인덱스 {idx}: {questions[idx]} → {result.content}")

  인덱스 1: 캐나다의 수도는? → 캐나다의 수도는 오타와(Ottawa)입니다.
  인덱스 2: 브라질의 수도는? → 브라질의 수도는 브라질리아입니다.
  인덱스 0: 대한민국의 수도는? → 대한민국의 수도는 서울입니다.
  인덱스 3: 호주의 수도는? → 호주의 수도는 캔베라(Canberra)입니다. 캔베라는 호주 정부의 공식적인 수도로, 국가의 주요 정치적 기관들이 위치하고 있습니다.


## 6. Tool Calling — 도구 호출

### 왜 Tool Calling이 필요한가요?

LLM은 학습 데이터에 기반해 답변하므로 **실시간 정보**(오늘 날씨, 최신 주가)를 알 수 없고, **정확한 계산**(복잡한 수식)도 보장하지 못해요. Tool Calling은 이 한계를 극복하기 위해 LLM이 "나 혼자 답하기 어려우니 외부 도구를 사용하겠다"고 판단하는 메커니즘이에요. 마치 의사가 진단을 내리기 위해 혈액 검사를 의뢰하는 것처럼, LLM도 필요한 정보를 외부에 요청할 수 있어요.

Tool Calling은 LLM이 외부 함수를 호출할 수 있게 해주는 기능이에요. 모델은 사용자 질문을 보고 어떤 도구를 호출할지, 어떤 인자를 넘길지 스스로 결정해요.

> 🎯 **강의 포인트**: Tool Calling은 LangGraph 에이전트의 핵심이에요. 다음 노트북인 `06-Tools-Integration`에서 `ToolNode`와 함께 실제 실행까지 연결하는 방법을 배워요. 여기서는 모델이 "어떤 도구를 어떻게 호출할지" 결정하는 부분을 이해해요.

> 🔑 **핵심 개념**: `bind_tools()`는 모델에 도구를 알려줘요. 모델이 도구 호출이 필요하다고 판단하면 `response.tool_calls`에 호출 정보가 채워지고, `response.content`는 빈 문자열이 돼요.

In [12]:
# ---------------------------------------------------
# Tool Calling: Pydantic 모델로 도구 정의
# ---------------------------------------------------
from pydantic import BaseModel, Field


# Pydantic BaseModel로 도구 스키마를 정의해요
# docstring이 도구의 용도를 모델에게 설명해요
class GetWeather(BaseModel):
    """Get the current weather in a given location."""

    location: str = Field(
        ...,
        description="도시와 국가명. 예: Seoul, South Korea"
    )


class SearchWeb(BaseModel):
    """Search the web for information on a given query."""

    query: str = Field(
        ...,
        description="검색할 질의어"
    )
    num_results: int = Field(
        default=5,
        description="반환할 최대 검색 결과 수"
    )


# bind_tools()로 모델에 도구를 바인딩해요
# 모델이 두 도구 중 적절한 것을 선택해서 호출해요
model_with_tools = model.bind_tools([GetWeather, SearchWeb])

# 날씨 관련 질문: GetWeather 도구를 선택할 거예요
response = model_with_tools.invoke("서울의 현재 날씨는 어때?")

# === content (비어 있어요: 도구 호출 시 텍스트 없음) ===
print(repr(response.content))
print()
# === tool_calls (도구 호출 정보) ===
print(response.tool_calls)

''

[{'name': 'GetWeather', 'args': {'location': 'Seoul, South Korea'}, 'id': 'call_Rm7Bdwdjyd4PvjlPJssdRRmK', 'type': 'tool_call'}]


In [13]:
# ---------------------------------------------------
# Tool Call 응답 구조 이해하기
# ---------------------------------------------------
# 도구 호출이 여러 개일 수도 있어요 (병렬 도구 호출)
if response.tool_calls:
    for tc in response.tool_calls:
        print(f"도구 이름: {tc['name']}")
        print(f"인자(args): {tc['args']}")
        print(f"호출 ID: {tc['id']}")
        # 이 id는 ToolMessage로 응답할 때 필요해요!
        print(f"타입: {tc['type']}")
        print()

도구 이름: GetWeather
인자(args): {'location': 'Seoul, South Korea'}
호출 ID: call_Rm7Bdwdjyd4PvjlPJssdRRmK
타입: tool_call



In [ ]:
# ============================================================
# 구현 예시: 계산기 도구를 만들어서 모델에 바인딩
# ============================================================
from pydantic import BaseModel, Field


class Calculate(BaseModel):
    """수학식을 계산해야 할 때 사용하세요."""

    expression: str = Field(description="계산할 수학식. 예: '15 + 27'")


model_calc = model.bind_tools([Calculate])
response_calc = model_calc.invoke("15 + 27을 계산해줘")

print("도구 호출 수:", len(response_calc.tool_calls))
for call in response_calc.tool_calls:
    print(call)


## 7. 구조화 출력: with_structured_output()

`with_structured_output()`은 모델의 응답을 미리 정의한 Pydantic 스키마에 맞게 파싱해요. 정보 추출, 데이터 파싱, API 응답 생성에 매우 유용해요.

> 💡 **실무 팁**: `with_structured_output()`은 내부적으로 두 가지 전략을 써요:
> - **ToolStrategy** (기본): Tool Calling을 이용해서 구조화된 응답 강제 (OpenAI, Anthropic 지원)
> - **ProviderStrategy**: 제공자의 네이티브 JSON 모드 사용
> 대부분의 경우 기본 ToolStrategy가 더 안정적이에요.

In [15]:
# ---------------------------------------------------
# with_structured_output(): 구조화된 응답 받기
# ---------------------------------------------------
from pydantic import BaseModel, Field
from typing import Optional


# 응답 스키마 정의: 추출하고 싶은 정보 구조를 정의해요
class PersonInfo(BaseModel):
    """사람의 기본 정보를 담는 스키마예요"""

    name: str = Field(description="사람의 이름")
    email: str = Field(description="이메일 주소")
    phone: Optional[str] = Field(
        default=None,
        description="전화번호 (없으면 None)"
    )
    age: Optional[int] = Field(
        default=None,
        description="나이 (없으면 None)"
    )


# with_structured_output()으로 모델을 래핑해요
# 이제 이 모델은 항상 PersonInfo 객체를 반환해요
structured_model = model.with_structured_output(PersonInfo)

# 비정형 텍스트에서 구조화된 정보 추출
text = """이름: 김철수, 이메일: chulsoo@example.com, 
전화번호: 010-1234-5678, 나이: 32세"""

result = structured_model.invoke(f"다음 텍스트에서 정보를 추출해주세요: {text}")

# 추출된 구조화 데이터:
print(result)
print()
print(f"이름: {result.name}")
print(f"이메일: {result.email}")
print(f"전화번호: {result.phone}")
print(f"나이: {result.age}")

name='김철수' email='chulsoo@example.com' phone='010-1234-5678' age=32

이름: 김철수
이메일: chulsoo@example.com
전화번호: 010-1234-5678
나이: 32


In [16]:
# ---------------------------------------------------
# 더 복잡한 구조화 출력 예시
# ---------------------------------------------------
from pydantic import BaseModel, Field
from typing import List, Literal


class SentimentAnalysis(BaseModel):
    """감성 분석 결과 스키마예요"""

    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="감성 분류: positive, negative, neutral 중 하나"
    )
    confidence: float = Field(
        description="분류 신뢰도 (0.0 ~ 1.0)"
    )
    key_phrases: List[str] = Field(
        description="핵심 감성 표현 목록 (최대 3개)"
    )
    summary: str = Field(
        description="감성 분석 요약 (한 문장)"
    )


sentiment_model = model.with_structured_output(SentimentAnalysis)

review = "이 제품은 정말 대단해요! 배송도 빠르고 품질도 최고예요. 완전 만족합니다."
analysis = sentiment_model.invoke(f"다음 리뷰를 분석해주세요: {review}")

# 감성 분석 결과:
print(f"  감성: {analysis.sentiment}")
print(f"  신뢰도: {analysis.confidence:.1%}")
print(f"  핵심 표현: {analysis.key_phrases}")
print(f"  요약: {analysis.summary}")

  감성: positive
  신뢰도: 95.0%
  핵심 표현: ['제품 대단해요', '배송 빠르고', '품질 최고', '완전 만족']
  요약: 이 리뷰는 제품에 대한 긍정적인 의견을 표현하고 있으며, 배송 속도와 품질에 대한 만족감을 나타냅니다.


## 8. 토큰 사용량 추적

대규모 애플리케이션에서는 비용 관리를 위해 토큰 사용량을 추적해야 해요. `UsageMetadataCallbackHandler`는 여러 모델 호출에 걸쳐 누적 사용량을 모델별로 자동 집계해요.

> 🎯 **강의 포인트**: 실무에서 가장 자주 받는 질문 중 하나가 "비용은 얼마나 드나요?"예요. 토큰 추적 패턴을 확실히 익혀두세요. `config` 파라미터를 통해 런타임에 주입되므로 기존 코드 수정 없이 추가할 수 있어요.

In [17]:
# ---------------------------------------------------
# UsageMetadataCallbackHandler: 토큰 사용량 추적
# ---------------------------------------------------
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import UsageMetadataCallbackHandler

# 두 가지 모델을 각각 초기화해요
model_gpt4o = init_chat_model("openai:gpt-4o-mini")    # 소형 모델
model_gpt4o_full = init_chat_model("openai:gpt-4o")    # 대형 모델 (주석 처리: 비용 절감)

# 콜백 핸들러를 생성해요
# 이 객체가 모든 호출의 토큰 사용량을 누적 집계해요
callback = UsageMetadataCallbackHandler()

# config의 callbacks 리스트에 핸들러를 추가해요
# 기존 코드 수정 없이 추적 기능을 추가할 수 있어요!
result_1 = model_gpt4o.invoke(
    "안녕하세요!",
    config={"callbacks": [callback]}
)
result_2 = model_gpt4o.invoke(
    "LangGraph란 무엇인가요?",
    config={"callbacks": [callback]}
)

# 누적 토큰 사용량 (모델별):
for model_name, usage in callback.usage_metadata.items():
    print(f"\n모델: {model_name}")
    print(f"  입력 토큰: {usage['input_tokens']}")
    print(f"  출력 토큰: {usage['output_tokens']}")
    print(f"  합계: {usage['total_tokens']}")


모델: gpt-4o-mini-2024-07-18
  입력 토큰: 24
  출력 토큰: 201
  합계: 225


In [18]:
# ---------------------------------------------------
# get_usage_metadata_callback: 컨텍스트 매니저 방식
# ---------------------------------------------------
# V1에서 추가된 컨텍스트 매니저 방식이에요
# with 블록 안에서의 모든 호출을 자동으로 추적해요
from langchain_core.callbacks import get_usage_metadata_callback

with get_usage_metadata_callback() as cb:
    # 이 블록 안의 모든 모델 호출을 추적해요
    r1 = model_gpt4o.invoke("1+1은?")
    r2 = model_gpt4o.invoke("2+2는?")
    r3 = model_gpt4o.invoke("3+3은?")

# 컨텍스트 매니저로 추적한 사용량:
for model_name, usage in cb.usage_metadata.items():
    print(f"모델: {model_name}")
    print(f"  총 입력 토큰: {usage['input_tokens']}")
    print(f"  총 출력 토큰: {usage['output_tokens']}")

모델: gpt-4o-mini-2024-07-18
  총 입력 토큰: 36
  총 출력 토큰: 27


## 9. config 파라미터 — 런타임 설정

`config` 파라미터를 통해 런타임에 모델 호출 동작을 세밀하게 제어할 수 있어요. LangSmith에서 실행을 추적하고 필터링하는 데 특히 유용해요.

> 💡 **실무 팁**: `run_name`과 `metadata`를 활용하면 LangSmith에서 특정 사용자나 세션의 실행을 쉽게 추적할 수 있어요. 프로덕션에서 디버깅할 때 필수예요.

| `config` 키 | 설명 |
|---|---|
| `run_name` | LangSmith 실행 이름 |
| `tags` | 분류·필터링용 태그 목록 |
| `metadata` | 임의의 키-값 메타데이터 (사용자 ID, 세션 등) |
| `callbacks` | 실행 중 호출될 콜백 핸들러 목록 |


In [19]:
# ---------------------------------------------------
# config 파라미터 사용
# ---------------------------------------------------

response_with_config = model.invoke(
    "LangChain V1의 주요 변경 사항을 한 줄로 설명해줘.",
    config={
        "run_name": "v1-explanation",             # LangSmith 실행 이름
        "tags": ["tutorial", "part02"],           # 분류 태그
        "metadata": {                              # 임의 메타데이터
            "user_id": "student_001",
            "session": "part02-models",
            "notebook": "02-Models.ipynb",
        },
    },
)

print(response_with_config.content)

LangChain V1에서는 모듈화된 구성 요소와 더 향상된 API를 통해 사용자 정의 및 확장성이 개선되었습니다.


## 10. 멀티모달 입력 — 이미지 처리

최신 LLM은 텍스트뿐 아니라 이미지도 처리할 수 있어요. LangChain V1에서는 `HumanMessage`의 `content`에 이미지 데이터를 포함시켜서 멀티모달 요청을 만들어요.

> 💡 **팁**: 외부 래퍼 클래스에 의존하지 않고 표준 LangChain V1 방식으로 직접 구현하는 것이 좋아요. 내부 동작을 이해할 수 있고 외부 의존성도 줄일 수 있어요.

In [20]:
# ---------------------------------------------------
# 멀티모달: URL 이미지 분석
# ---------------------------------------------------
# 표준 LangChain V1 멀티모달 방식을 사용해요
# HumanMessage의 content를 리스트로 만들어서 이미지를 포함시켜요
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

# 멀티모달 모델 초기화 (gpt-4o-mini는 비전 지원)
vision_model = init_chat_model("openai:gpt-4o-mini")

# 이미지 URL을 포함한 메시지 구성
# content를 리스트로 구성할 때:
# - {"type": "text", "text": "..."}: 텍스트 블록
# - {"type": "image_url", "image_url": {"url": "..."}}: 이미지 URL 블록
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d9/Collage_of_Nine_Dogs.jpg/1280px-Collage_of_Nine_Dogs.jpg"

multimodal_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "이 이미지에 무엇이 있나요? 한국어로 간략하게 설명해주세요."
        },
        {
            "type": "image_url",
            "image_url": {"url": IMAGE_URL}
        },
    ]
)

# 멀티모달 요청 실행
# 이미지 분석 결과:
for chunk in vision_model.stream([multimodal_message]):
    print(chunk.content, end="", flush=True)
print()

이 이미지는 다양한 품종의 개들이 모여 있는 모습입니다. 각 개들은 서로 다른 색상, 크기, 털의 길이를 가지고 있으며, 총 9개의 사진이 3x3 형태로 배열되어 있습니다. 개들은 잔디 위에 누워 있거나 앉아 있거나 서 있는 모습으로, 각각의 특징이 잘 드러나 있습니다.


In [21]:
# ---------------------------------------------------
# 멀티모달 헬퍼 함수 직접 구현
# ---------------------------------------------------
# 멀티모달 이미지 입력을 처리하는 함수예요

def analyze_image(
    image_url: str,
    question: str = "이 이미지를 설명해주세요.",
    system_prompt: str = "You are a helpful image analysis assistant. Answer in Korean.",
    model_name: str = "openai:gpt-4o-mini"
) -> str:
    """이미지 URL을 받아서 분석 결과를 반환하는 함수예요.
    
    Args:
        image_url: 분석할 이미지의 URL
        question: 이미지에 대한 질문
        system_prompt: 시스템 프롬프트
        model_name: 사용할 비전 모델
    
    Returns:
        이미지 분석 결과 텍스트
    """
    from langchain.chat_models import init_chat_model
    from langchain.messages import HumanMessage, SystemMessage
    
    # 비전 모델 초기화
    llm = init_chat_model(model_name)
    
    # 시스템 + 멀티모달 메시지 구성
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(
            content=[
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]
        ),
    ]
    
    # 스트리밍으로 실시간 출력
    result = ""
    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
        result += chunk.content
    print()
    return result


# 함수 테스트
result = analyze_image(
    image_url=IMAGE_URL,
    question="이미지에 보이는 동물들의 종류와 특징을 간략히 설명해주세요."
)

이미지에 보이는 동물들은 다양한 종류의 개들입니다. 각 개의 특징을 간략히 설명하겠습니다.

1. **골든 리트리버**: 중대형견으로, 금색의 아름다운 털을 가진 친화적인 성격의 개입니다. 매우 지적이고 훈련이 잘됩니다.

2. **푸들**: 다양한 크기로 존재하며, 곱슬곱슬한 털이 특징입니다. 지능이 높고, 활발한 성격을 가지고 있습니다.

3. **요크셔 테리어**: 소형견으로, 긴 털과 귀여운 외모가 특징입니다. 활발하고 용감한 성격입니다.

4. **복서**: 단단한 체격을 가진 중형견으로, 활발하고 운동량이 많은 품종입니다. 보호자에게 매우 충성스럽습니다.

5. **포메라니안**: 소형견으로, 부풀어 오른 듯한 털과 귀여운 외모를 가집니다. 활기차고 사교적인 성격입니다.

6. **비글**: 중소형견으로, 짧은 털과 긴 귀가 특징입니다. 매우 뛰어난 후각을 가지고 있으며, 온순한 성격입니다.

7. **바셋 하운드**: 길고 낮은 다리와 귀가 긴 특징으로, 느긋한 성격을 가집니다. 후각이 뛰어나고 사냥 개로 인기가 많습니다.

8. **뉴펀들랜드**: 대형견으로, 두꺼운 털과 강한 체격을 가지고 있습니다. 수영을 잘하고, 가족에게 매우 헌신적입니다.

이들 개들은 저마다 다양한 성격과 특성을 가지고 있습니다.


## 11. Logprobs — 토큰 확률 분포

`logprobs` 옵션을 활성화하면 모델이 각 토큰을 생성할 때의 로그 확률을 받을 수 있어요. 모델의 확신도를 측정하거나 불확실성을 분석할 때 유용해요.

> 🔑 **핵심 개념**: Logprobs는 현재 OpenAI 모델에서만 지원해요. `bind(logprobs=True)`로 활성화하면 응답 메타데이터에 토큰별 로그 확률이 포함돼요. 로그 확률을 확률로 변환하려면 `math.exp(logprob)` 또는 `math.exp(logprob) * 100` (%)로 계산해요.

In [22]:
# ---------------------------------------------------
# Logprobs: 토큰 확률 분포 확인
# ---------------------------------------------------
# 토큰 확률을 추출하는 함수를 직접 구현해요
import math
from langchain_openai import ChatOpenAI


def extract_token_probabilities(response) -> dict:
    """AIMessage에서 토큰별 확률을 추출하는 함수예요.
    
    logprobs가 활성화된 모델의 응답에서만 동작해요.
    
    Args:
        response: logprobs가 포함된 AIMessage 객체
    
    Returns:
        {'tokens': [...], 'probabilities': [...]} 딕셔너리
    """
    logprobs_data = response.response_metadata.get("logprobs", {})
    if not logprobs_data or "content" not in logprobs_data:
        return {"tokens": [], "probabilities": []}
    
    tokens = []
    probabilities = []
    
    for token_info in logprobs_data["content"]:
        token = token_info["token"]
        logprob = token_info["logprob"]
        prob_percent = round(math.exp(logprob) * 100, 2)  # 퍼센트로 변환
        tokens.append(token)
        probabilities.append(prob_percent)
    
    return {"tokens": tokens, "probabilities": probabilities}


# logprobs=True로 바인딩: OpenAI 모델에서만 지원해요
logprobs_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,  # 확정적인 답변을 원할 때
).bind(logprobs=True)

# 예/아니오 질문으로 모델의 확신도 측정
response_lp = logprobs_model.invoke(
    "대한민국의 수도는 부산입니까? '예' 또는 '아니오'로만 답해주세요."
)

print(f"응답: {response_lp.content}")
print()

# 토큰 확률 추출
prob_data = extract_token_probabilities(response_lp)

if prob_data["tokens"]:
    # 토큰별 확률 분포:
    for token, prob in zip(prob_data["tokens"], prob_data["probabilities"]):
        bar = "█" * int(prob / 5)  # 간단한 막대 그래프
        print(f"  {token!r:15} | {bar} {prob:.1f}%")
else:
    # logprobs 데이터가 없어요.
    pass

응답: 아니오.

  '아'             | ████████████████████ 100.0%
  '니'             | ████████████████████ 100.0%
  '오'             | ███████████████████ 98.9%
  '.'             | ███████████████████ 98.4%


## 12. 커스텀/사내 모델 서버 연결 전략

이 과정에서는 별도 로컬 런타임 도구 실습을 다루지 않아요. 커스텀 엔드포인트나 사내 모델 서버가 필요하면 다음 섹션의 **OpenAI 호환 API** 패턴처럼 `base_url`을 지정해 연결합니다.

> 💡 **실무 팁**: 제공자가 OpenAI 호환 Chat Completions API를 제공하면 `ChatOpenAI(base_url=..., api_key=...)` 형태로 같은 LangChain 인터페이스를 유지할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 커스텀 모델 서버 연결 안내
# ---------------------------------------------------
# 이 과정에서는 별도 로컬 런타임 도구 실습을 생략해요.
# 커스텀 엔드포인트나 사내 모델 서버는 다음 OpenAI 호환 API 섹션에서 다룹니다.

print("커스텀/사내 모델 서버는 다음 OpenAI 호환 API 예제를 사용하세요.")


## 13. OpenAI 호환 API — 커스텀 엔드포인트

`ChatOpenAI`의 `base_url`을 변경하면 LM Studio, vLLM, OpenRouter 등 OpenAI API 형식을 따르는 서비스에 연결할 수 있어요.

> 🎯 **강의 포인트**: 이 패턴은 회사의 자체 LLM 서버나 비용 효율적인 OpenRouter 같은 서비스를 사용할 때 매우 유용해요. `base_url`만 바꾸면 나머지 코드는 그대로예요.

### 파라미터 전달 방식

| 방식 | 용도 | 예시 |
|------|------|------|
| `model_kwargs` | 표준 OpenAI 파라미터 | `max_completion_tokens`, `stream_options` |
| `extra_body` | 제공자 고유 파라미터 | vLLM의 `use_beam_search`, LM Studio의 `ttl` |

In [ ]:
# ---------------------------------------------------
# OpenAI 호환 API: base_url 변경으로 다양한 서비스 연결
# ---------------------------------------------------
from langchain_openai import ChatOpenAI

# LM Studio 연결 예시 (로컬 GUI LLM 서버)
# lm_studio_model = ChatOpenAI(
#     base_url="http://localhost:1234/v1",    # LM Studio 기본 주소
#     api_key="lm-studio",                    # 아무 문자열이나 가능
#     model="mlx-community/Llama-3.2-3B-Instruct",
#     extra_body={"ttl": 300}                 # LM Studio 고유 파라미터
# )

# vLLM 연결 예시 (고성능 추론 서버)
# vllm_model = ChatOpenAI(
#     base_url="http://localhost:8000/v1",    # vLLM 기본 주소
#     api_key="EMPTY",                        # vLLM은 API 키 불필요
#     model="meta-llama/Llama-2-7b-chat-hf",
#     extra_body={                            # vLLM 고유 파라미터
#         "use_beam_search": True,
#         "best_of": 4
#     }
# )

# OpenRouter 연결 예시 (다양한 모델 통합 서비스)
# openrouter_model = ChatOpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key="sk-or-...",                   # OpenRouter API 키
#     model="anthropic/claude-3-haiku",      # 모델명 형식이 달라요
# )

# model_kwargs vs extra_body 구분 예시
# model_kwargs: 표준 OpenAI API 파라미터 (최상위 payload에 병합)
# model_with_kwargs = ChatOpenAI(
#     model="gpt-4o-mini",
#     model_kwargs={
#         "stream_options": {"include_usage": True},
#         "max_completion_tokens": 300,
#     }
# )

# OpenAI 호환 API 설정 예시 코드입니다.
# 주석을 해제하고 실제 서버 주소로 변경하면 바로 사용할 수 있어요.

OpenAI 호환 API 설정 예시 코드입니다.
주석을 해제하고 실제 서버 주소로 변경하면 바로 사용할 수 있어요.


## 14. reasoning_effort — 추론 강도 조절

OpenAI의 추론 모델(o1, o3 등)은 `reasoning_effort`로 추론 깊이를 조절할 수 있어요. 비용과 품질 사이의 균형을 맞출 수 있어요.

> 🔑 **핵심 개념**: `reasoning_effort`는 추론 모델의 "생각하는 시간"을 조절해요. `high`는 더 깊이 생각해서 정확도가 높지만 느리고 비싸요. 수학, 코딩, 논리적 추론 문제에 적합해요.

| `reasoning_effort` | 설명 | 권장 용도 |
|---|---|---|
| `minimal` | 최소 추론 (가장 빠름) | 단순 질문 |
| `low` | 낮은 추론 | 가벼운 작업 |
| `medium` | 중간 추론 (균형) | 일반적 작업 |
| `high` | 깊은 추론 | 수학·코딩·복잡한 로직 |


In [25]:
# ---------------------------------------------------
# reasoning_effort: 추론 강도 조절
# ---------------------------------------------------
# reasoning_effort는 OpenAI 추론 모델(o1, o3 등)에서 지원해요
# gpt-4o-mini는 지원하지 않으므로 아래는 참고 코드예요
from langchain_openai import ChatOpenAI


# o3-mini 모델 예시 (추론 모델)
reasoning_model = ChatOpenAI(
    model="o3-mini",
    reasoning_effort="medium",
)

# gpt-4o-mini로 대신 테스트 (추론 강도 없음)
# reasoning_model = ChatOpenAI(
#     model="gpt-4o-mini",
#     temperature=0.1,
# )

# 스트리밍으로 응답 받기
# 복잡한 추론 문제:
for chunk in reasoning_model.stream(
    "다음 수열의 규칙을 찾고 다음 수를 예측해주세요: 2, 6, 12, 20, 30, 42, ?"
):
    print(chunk.content, end="", flush=True)
print()

이 수열은 각 항이 n*(n+1)의 형태를 가집니다.

예를 들어, 
첫 번째 항: 1×2 = 2  
두 번째 항: 2×3 = 6  
세 번째 항: 3×4 = 12  
네 번째 항: 4×5 = 20  
다섯 번째 항: 5×6 = 30  
여섯 번째 항: 6×7 = 42  

따라서, 일곱 번째 항은 7×8 = 56이 됩니다.

즉, 다음 수는 56입니다.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`init_chat_model`**: `"provider:model"` 형식으로 20+ 제공자를 통일된 방식으로 초기화해요. V1의 가장 중요한 변화예요
- **5가지 호출 방식**: `invoke`(동기), `stream`(실시간), `ainvoke`/`astream`(비동기), `batch`(병렬)을 상황에 맞게 선택해요
- **Tool Calling**: `bind_tools()`로 모델에 도구를 알려주고, `response.tool_calls`로 호출 정보를 받아요
- **구조화 출력**: `with_structured_output(PydanticModel)`으로 일관된 구조의 응답을 보장해요
- **토큰 추적**: `UsageMetadataCallbackHandler`나 `get_usage_metadata_callback`으로 비용을 모델별로 집계해요
- **멀티모달**: `HumanMessage`의 `content`에 이미지 블록을 포함시켜서 이미지를 분석해요
- **Logprobs**: `bind(logprobs=True)`로 토큰 확률 분포를 얻어서 모델의 확신도를 측정해요
- **OpenAI 호환 API**: `base_url` 변경으로 LM Studio, vLLM, OpenRouter 등 다양한 서버에 연결해요

## 다음 노트북 예고

다음 `03-Messages.ipynb`에서는 **메시지 타입과 Content Blocks**를 배워요. `HumanMessage`, `AIMessage`, `SystemMessage`, `ToolMessage` 4가지 메시지 타입의 구조와 멀티모달 Content Blocks를 활용하는 방법을 자세히 다뤄요.